In [2]:
# --- Core libraries ---
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import geopandas as gpd
from shapely.geometry import Point
import plotly.express as px # for interactive plots

# --- Scikit-learn: model training, preprocessing, CV, evaluation ---
from sklearn.model_selection import (
    GroupKFold,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
    GridSearchCV
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    balanced_accuracy_score,
    roc_auc_score,
    make_scorer,
    ConfusionMatrixDisplay
)
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import BallTree

# --- XGBoost ---
from xgboost import XGBRegressor, plot_tree, plot_importance
import xgboost  # Optional: useful for checking version

# --- Other ---
from haversine import haversine
import scipy.stats as stats
import gstools as gs #variogram modeling
from sklearn.metrics import pairwise_distances # for distance calculations
from scipy.cluster.hierarchy import linkage, dendrogram #for hierarchical clustering dendrogram
from tabulate import tabulate # for pretty printing tables


#--- Custom Functions as defined by the user ---
import sys
sys.path.append(os.path.abspath("tools"))
import functions

In [20]:
print("Current working directory:", os.getcwd())

#load in 
df = pd.read_csv("../../../data/finaldatasets/testdata/finalfr/AlphaTree.csv")  # for variogram, etc.

Current working directory: /home/sebastian-dohne/Documents/FinalProject/code/analysis/kagglefeatureengineering


/tmp/ipykernel_161815/2963851490.py:4: DtypeWarning: Columns (6,7,12,13,14,15,16,17,20,22,23,29,31,33,34,35,36,37,44,45,51,52,56,57,58) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../../data/finaldatasets/testdata/finalfr/AlphaTree.csv")  # for variogram, etc.


Data summary function

In [21]:
functions.summarize_dataframe(df)

| variable                                              | dtype   |   count |   pct_complete |   n_unique |
|-------------------------------------------------------|---------|---------|----------------|------------|
| id                                                    | int64   |   69430 |          100   |      69036 |
| EvapS3                                                | float64 |   69430 |          100   |       1155 |
| EvapS1                                                | float64 |   69430 |          100   |       1155 |
| pre9                                                  | float64 |   69428 |          100   |        639 |
| pre8                                                  | float64 |   69428 |          100   |        632 |
| pre7                                                  | float64 |   69428 |          100   |        650 |
| pre6                                                  | float64 |   69428 |          100   |        674 |
| pre5                      

#### Feature engineering Values in dataset

    Complete feature engineering for wheat yield prediction.
    Transforms 169 raw features into ~50 biologically meaningful features.
    
    Parameters:
    df (pandas.DataFrame): Raw wheat dataset
    
    Returns:
    pandas.DataFrame: Feature engineered dataset

In [ ]:
print("Starting wheat feature engineering...")
print(f"Original dataset shape: {df.shape}")

# Create copy to avoid modifying original data
df_eng = df.copy()

# ==================== TEMPERATURE FEATURES ====================
print("\n1. Engineering temperature features...")

# Average temperature during early establishment (months 1-2)
# Critical for germination and early root development

temp_cols = [f'temp{i}' for i in range(1, 10)]


df_eng['temp_establishment'] = df[['temp1', 'temp2']].mean(axis=1)

# Average temperature during stem elongation (months 3-4) 
# Rapid biomass accumulation phase, temperature affects development rate
df_eng['temp_stem_elongation'] = df[['temp2', 'temp3', 'temp4']].mean(axis=1)

# Average temperature during critical yield determination period (months 4-6)
# When grain number is determined - most sensitive period for yield
df_eng['temp_critical_period'] = df[['temp4', 'temp5', 'temp6']].mean(axis=1)

# Average temperature during grain filling (months 6-9)
# Affects grain weight and final yield quality
df_eng['temp_grain_filling'] = df[['temp6','temp7', 'temp8', 'temp9']].mean(axis=1)

# Count of heat stress months (mean >30°C) during grain filling
# Heat stress during grain filling reduces yield significantly
temp_grain_cols = ['temp6','temp7', 'temp8']
df_eng['num_heat_stress_months'] = df[temp_grain_cols].apply(
    lambda row: (row > 30).mean(), axis=1
)

# Temperature above optimal during grain filling (optimal ~20-25°C for wheat)
df_eng['temp_above_optimal_grain_fill'] = df[['temp6', 'temp7', 'temp8', 'temp9']].apply(
    lambda row: np.maximum(row - 25, 0).mean(), axis=1
)


Starting wheat feature engineering...
Original dataset shape: (69430, 169)

1. Engineering temperature features...


In [28]:
# ==================== PRECIPITATION FEATURES ====================
print("2. Engineering precipitation features...")

precip_cols = [f'pre{i}' for i in range(1, 10)]

# Total precipitation during establishment (months 1-2)
# Critical for seed germination and emergence
df_eng['precip_establishment'] = df[['pre1', 'pre2']].sum(axis=1)

# Total precipitation during vegetative growth (months 3-4)
# Supports tillering and biomass accumulation
df_eng['precip_vegetative'] = df[['pre3', 'pre4']].sum(axis=1)

# Total precipitation during reproductive phase (months 5-6)
# Most drought-sensitive period, critical for grain number
df_eng['precip_reproductive'] = df[['pre5', 'pre6']].sum(axis=1)

# Total precipitation during grain filling (months 7-9)
# Affects final grain weight and yield security
df_eng['precip_grain_filling'] = df[['pre7', 'pre8', 'pre9']].sum(axis=1)

# Total seasonal precipitation
# Overall water availability indicator
precip_cols = [f'pre{i}' for i in range(1, 10)]
df_eng['precip_total_season'] = df[precip_cols].sum(axis=1)


2. Engineering precipitation features...


In [44]:
# ==================== VEGETATION INDICES ====================

# ==================== NDVI FEATURES ====================
print("3. Engineering NDVI features...")

# Create list of all NDVI column names (months 1-9)
ndvi_cols = [f'NDVI{i}' for i in range(1, 10)]

# Maximum NDVI across the entire growing season
# Indicates peak canopy development and photosynthetic potential
# Higher values = better biomass accumulation and light interception
df_eng['NDVI_max'] = df[ndvi_cols].max(axis=1)

# Average NDVI during critical yield determination period (months 4-6)
# This is when grain number is set - most important period for final yield
# Canopy health during this phase directly impacts grain number per spike
df_eng['NDVI_critical_avg'] = df[['NDVI4', 'NDVI5', 'NDVI6']].mean(axis=1)

# Early season NDVI growth rate (change from month 1 to 3)
# How quickly the canopy establishes after emergence
# Steep positive slope = vigorous early growth and good tillering
df_eng['NDVI_early_slope'] = (df['NDVI3'] - df['NDVI1']) / 2

# Late season NDVI decline rate (change from month 7 to 9)
# How fast the canopy senesces during grain filling
# Less negative slope = better "stay-green" trait, longer grain filling period
df_eng['NDVI_late_slope'] = (df['NDVI9'] - df['NDVI7']) / 2

# Average NDVI during grain filling period (months 7-9)
# Canopy health when final grain weight is determined
# Higher values = better photosynthetic capacity for grain filling
df_eng['NDVI_grain_filling'] = df[['NDVI7', 'NDVI8', 'NDVI9']].mean(axis=1)

# NDVI maintenance ratio - stay-green characteristic
# What fraction of peak greenness is maintained during grain filling
# Higher ratio = better drought tolerance and extended grain filling
# +1e-6 prevents division by zero errors
df_eng['NDVI_maintenance_ratio'] = df_eng['NDVI_grain_filling'] / (df_eng['NDVI_max'] + 1e-6)

# Total NDVI accumulated over the season (area under the curve)
# Proxy for total seasonal photosynthetic capacity
# Higher sum = more accumulated biomass over entire growing season
df_eng['NDVI_area_under_curve'] = df[ndvi_cols].sum(axis=1)


3. Engineering NDVI features...


In [45]:
# ==================== EVI FEATURES (IDENTICAL STRUCTURE) ====================
print("4. Engineering EVI features...")

# Create list of all EVI column names (months 1-9)
evi_cols = [f'evi_{i}' for i in range(1, 10)]

# Maximum EVI across the entire growing season
# Like NDVI_max but less prone to saturation at high biomass levels
# Better for distinguishing between good vs excellent crop conditions
df_eng['EVI_max'] = df[evi_cols].max(axis=1)

# Average EVI during critical yield determination period (months 4-6)
# EVI version of critical period canopy health
# More sensitive than NDVI for dense canopies during peak growth
df_eng['EVI_critical_avg'] = df[['evi_4', 'evi_5', 'evi_6']].mean(axis=1)

# Early season EVI growth rate (change from month 1 to 3)
# EVI-based measure of establishment vigor
# May capture early growth patterns that NDVI misses due to soil effects
df_eng['EVI_early_slope'] = (df['evi_3'] - df['evi_1']) / 2

# Late season EVI decline rate (change from month 7 to 9)
# EVI-based senescence rate during grain filling
# Less affected by atmospheric conditions than NDVI slope
df_eng['EVI_late_slope'] = (df['evi_9'] - df['evi_7']) / 2

# Average EVI during grain filling period (months 7-9)
# EVI-based canopy health during final yield formation
# Better discrimination of canopy quality at high biomass levels
df_eng['EVI_grain_filling'] = df[['evi_7', 'evi_8', 'evi_9']].mean(axis=1)

# EVI maintenance ratio - stay-green characteristic
# EVI version of photosynthetic maintenance during grain filling
# May be more accurate than NDVI ratio for dense, healthy canopies
# +1e-6 prevents division by zero errors
df_eng['EVI_maintenance_ratio'] = df_eng['EVI_grain_filling'] / (df_eng['EVI_max'] + 1e-6)

# Total EVI accumulated over the season (area under the curve)
# EVI-based proxy for total seasonal photosynthetic capacity
# Less saturated measure of total biomass accumulation than NDVI sum
df_eng['EVI_area_under_curve'] = df[evi_cols].sum(axis=1)


4. Engineering EVI features...


In [46]:
# ==================== LEAF AREA INDEX (LAI) ====================
print("5. Engineering LAI features...")

lai_cols = [f'lai_month_{i}' for i in range(1, 8)]
df_eng[lai_cols] = df[lai_cols]

# Maximum LAI - peak leaf area development
df_eng['LAI_max'] = df[lai_cols].max(axis=1)

# LAI during early establishment (months 1-2) - critical for tillering
# Early LAI indicates how well the crop establishes and starts photosynthesis
df_eng['LAI_early_establishment'] = df[['lai_month_1', 'lai_month_2']].mean(axis=1)

# LAI during mid-season (months 4-6) - peak biomass accumulation
# When the crop reaches maximum photosynthetic capacity
df_eng['LAI_mid_season'] = df[['lai_month_3', 'lai_month_4', 'lai_month_5']].mean(axis=1)

# LAI during late season/grain filling (months 6-7) - stay-green trait
# Maintaining leaf area during grain filling is crucial for grain weight
df_eng['LAI_late_season'] = df[['lai_month_6', 'lai_month_7']].mean(axis=1)

# LAI growth rate during establishment (early vigor)
# How quickly leaf area develops in first 2 months - indicates crop vigor
df_eng['LAI_early_growth_rate'] = (df['lai_month_2'] - df['lai_month_1']) / 1

# LAI stay-green index - maintenance of leaf area during grain filling
# Higher values = better photosynthetic maintenance during critical period
df_eng['LAI_stay_green_index'] = df_eng['LAI_late_season'] / (df_eng['LAI_max'] + 1e-6)

# LAI maintenance duration
# Count months where LAI > 80% of maximum
lai_data = df[lai_cols].values  # Get as numpy array
lai_max_vals = np.nanmax(lai_data, axis=1, keepdims=True)  # Max for each row
threshold = 0.8 * lai_max_vals
df_eng['LAI_duration_high'] = np.sum(lai_data >= threshold, axis=1)

# LAI decline rate - SIMPLIFIED
# Simple decline from max to final month
df_eng['LAI_decline_rate'] = (df['lai_month_7'] - df_eng['LAI_max']) / 3

5. Engineering LAI features...


/tmp/ipykernel_161815/2397759875.py:33: RuntimeWarning: All-NaN slice encountered
  lai_max_vals = np.nanmax(lai_data, axis=1, keepdims=True)  # Max for each row


In [47]:
# ==================== SOIL MOISTURE ====================
print("6. Engineering soil moisture features...")

# Soil moisture during different growth phases
# Critical for understanding water stress timing

# Early season soil moisture (establishment)
df_eng['soil_moisture_establishment'] = df[['SoilM1', 'SoilM2']].mean(axis=1)

# Vegetative growth soil moisture 
df_eng['soil_moisture_vegetative'] = df[['SoilM3', 'SoilM4']].mean(axis=1)

# Critical period soil moisture (most sensitive to drought)
df_eng['soil_moisture_critical'] = df[['SoilM5', 'SoilM6']].mean(axis=1)

# Grain filling soil moisture
df_eng['soil_moisture_grain_fill'] = df[['SoilM7', 'SoilM8', 'SoilM9']].mean(axis=1)

# Average seasonal soil moisture
soilm_cols = [f'SoilM{i}' for i in range(1, 10)]
df_eng['soil_moisture_seasonal_avg'] = df[soilm_cols].mean(axis=1)

# Soil moisture variability (consistency of water supply)
df_eng['soil_moisture_variability'] = df[soilm_cols].std(axis=1)


6. Engineering soil moisture features...


In [ ]:
# ==================== EVAPOTRANSPIRATION and Evapstress ====================
print("7. Engineering evapotranspiration features...")

evap_cols = [f'Evap{i}' for i in range(1, 10)]
evaps_cols = [f'EvapS{i}' for i in range(1, 10)]

# Total seasonal evapotranspiration - water demand
df_eng['evap_total_season'] = df[evap_cols].sum(axis=1)
df_eng['evapS_total_season'] = df[evaps_cols].sum(axis=1)

# Evapotranspiration during early life - peak water demand
df_eng['evap_early_life'] = df[['Evap1', 'Evap2', 'Evap3']].mean(axis=1)
df_eng['evapS_early_life'] = df[['EvapS1', 'EvapS2', 'EvapS3']].mean(axis=1)

# Evapotranspiration during critical period - peak water demand
df_eng['evap_critical_period'] = df[['Evap4', 'Evap5', 'Evap6']].mean(axis=1)
df_eng['evapS_critical_period'] = df[['EvapS4', 'EvapS5', 'EvapS6']].mean(axis=1)

# Water availability - evapotranspiration relative to precipitation
# Lower values indicate more efficient water use
df_eng['water_availibility'] = df_eng['evap_total_season'] / (df_eng['precip_total_season'] + 1e-6)
df_eng['water_use_efficiency_surface'] = df_eng['evapS_total_season'] / (df_eng['precip_total_season'] + 1e-6)


7. Engineering evapotranspiration features...


In [49]:
# ==================== TRANSPIRATION ====================
print("8. Engineering transpiration features...")

trans_cols = [f'Trans{i}' for i in range(1, 10)]

# Peak transpiration - maximum plant water use
df_eng['transpiration_peak'] = df[trans_cols].max(axis=1)

# Transpiration during early stage period
df_eng['transpiration_early_stage'] = df[['Trans1', 'Trans2', 'Trans3']].mean(axis=1)

# Transpiration during critical period
df_eng['transpiration_critical'] = df[['Trans4', 'Trans5', 'Trans6']].mean(axis=1)

# Total seasonal transpiration - total plant water use
df_eng['transpiration_total'] = df[trans_cols].sum(axis=1)


8. Engineering transpiration features...


In [50]:
# ==================== FPAR (Fraction Photosynthetically Active Radiation) ====================
print("9. Engineering FPAR features...")

fpar_cols = [f'fpar_month_{i}' for i in range(1, 8)]

# Maximum FPAR - peak light interception
df_eng['FPAR_max'] = df[fpar_cols].max(axis=1)

# FPAR during critical period - light interception when yield is determined
df_eng['FPAR_critical'] = df[['fpar_month_4', 'fpar_month_5']].mean(axis=1)

# Early season FPAR - establishment vigor
df_eng['FPAR_early'] = df[['fpar_month_1', 'fpar_month_2', 'fpar_month_3']].mean(axis=1)

# 3. FPAR MAINTENANCE RATIO - Stay-green light interception trait
# How well light capture is maintained during grain filling
# Critical for stress tolerance and final grain weight
fpar_max = df[fpar_cols].max(axis=1)
fpar_late = df[['fpar_month_6', 'fpar_month_7']].mean(axis=1)
df_eng['FPAR_maintenance_ratio'] = fpar_late / (fpar_max + 1e-6)

# 2. FPAR AREA UNDER CURVE - Total seasonal photosynthetic capacity  
# Integrates light capture across entire growing season
# Best single predictor of biomass accumulation and yield potential
df_eng['FPAR_area_under_curve'] = df[fpar_cols].sum(axis=1)

9. Engineering FPAR features...


In [51]:
# ==================== STRESS INDICATORS & RATIOS ====================
print("10. Creating stress indicators and biological ratios...")

# Water stress index - water demand vs supply during critical period
# Higher values indicate more water stress
df_eng['water_stress_index'] = df_eng['evap_critical_period'] / (df_eng['precip_reproductive'] + 1e-6)

# Drought stress during grain filling
df_eng['drought_stress_grain_fill'] = df_eng['evap_total_season'] / (df_eng['precip_grain_filling'] + 1e-6)


# Canopy vigor maintenance (how well vegetation indices are maintained)
df_eng['canopy_vigor_maintenance'] = (df_eng['NDVI_grain_filling'] + df_eng['EVI_grain_filling']) / 2

# Light use efficiency proxy
df_eng['light_use_efficiency'] = df_eng['NDVI_max'] / (df_eng['FPAR_max'] + 1e-6)

# Biomass accumulation rate proxy (early NDVI slope)
df_eng['biomass_accumulation_rate'] = df_eng['NDVI_early_slope']

# Stay-green index (maintenance of photosynthetic activity)
df_eng['stay_green_index'] = df_eng['NDVI_maintenance_ratio'] * df_eng['LAI_duration_high']

10. Creating stress indicators and biological ratios...


In [52]:
# ==================== MANAGEMENT and Human influenced FEATURES ====================
print("11. Selecting management features...")

# Farm management practices that can be controlled by farmers
management_features = [
    'N.rate..kg.N.ha.1.',               # Nitrogen fertilizer application rate
    'P.rate..kg.P.ha.1.',               # Phosphorus fertilizer application rate  
    'pr_irrigated',                    # Irrigation water applied
    'gdp_per_capita',                    # Economic development indicator
    'Agricultural_Use_._Fungicides_and_Bactericides_.t.',  # Fungicide and bactericide use
    'Use_per_area_of_cropland_._Pesticides_.total._.kg.ha.', # Pesticide use per area
    'Agricultural_Use_._Herbicides_.t.',  # Herbicide use
    'Agricultural_Use_._Insecticides_.t.',  # Insecticide use
    'Crop.variety',                # Crop variety used
    'Tillage.type',                # Tillage method used
]

# Only keep management features that exist in the dataset
existing_mgmt_features = [f for f in management_features if f in df_eng.columns]
print(f"Keeping {len(existing_mgmt_features)} management features")

11. Selecting management features...
Keeping 10 management features


In [53]:
# ==================== ENVIRONMENTAL FEATURES ====================
print("12. Selecting environmental features...")

# Environmental conditions/ location features that cannot be controlled by farmers
environmental_features = [
    'Sand', 'Silt',                     # Soil texture - affects water/nutrient retention
    'Soil.pH',                          # Soil acidity - affects nutrient availability
    'Soil.organic.carbon..g.C.kg.1.',   # Soil organic matter - fertility indicator
    'Soil_N',                            # Soil nitrogen content - affects fertility
    'Soil_type',                        # Soil type - affects water/nutrient retention
    'Location',
    'State.Region.County.Province',
    'id',
    'start_date',
    'end_date',
    'Continent',
    'Country',
    'year',
    'sowing_year',
    'Elevation',                        # Altitude - affects temperature and rainfall
    'Conversion.for.latitude',          # Geographic latitude - affects climate
    'Conversion.for.longitude',         # Geographic longitude - affects climate
    'AEZ',
    'Pest.prescence....64.',            # Binary Pest presence indicator
]

# Only keep environmental features that exist in the dataset
existing_env_features = [f for f in environmental_features if f in df_eng.columns]
print(f"Keeping {len(existing_env_features)} environmental features")

12. Selecting environmental features...
Keeping 18 environmental features


In [54]:
# ==================== ENGINEERED FEATURES ====================
# Get all engineered features (columns that were created during feature engineering)
original_columns = set(df.columns)  # Original dataset columns
engineered_columns = [col for col in df_eng.columns if col not in original_columns]
print(f"Found {len(engineered_columns)} engineered features")

# Target variable
target = 'Grain.yield..tons.ha.1.'

Found 62 engineered features


In [55]:
# ==================== COMBINE ALL DESIRED FEATURES ====================
# Create the final feature list (avoid duplicates using set)
all_desired_features = set(
    existing_mgmt_features + 
    existing_env_features + 
    engineered_columns + 
    [target]
)

# Convert back to list and only keep features that exist in df_eng
final_feature_list = [f for f in all_desired_features if f in df_eng.columns]

print(f"\n=== FEATURE SELECTION SUMMARY ===")
print(f"Management features: {len(existing_mgmt_features)}")
print(f"Environmental features: {len(existing_env_features)}")
print(f"Engineered features: {len(engineered_columns)}")
print(f"Target variable: 1")
print(f"Total desired features: {len(final_feature_list)}")


=== FEATURE SELECTION SUMMARY ===
Management features: 10
Environmental features: 18
Engineered features: 62
Target variable: 1
Total desired features: 91


In [56]:
# ==================== CREATE FINAL DATASET ====================
# 🔧 FIX: Create DataFrame with only selected features
df_final = df_eng[final_feature_list].copy()

print(f"\n=== FEATURE ENGINEERING COMPLETE ===")
print(f"Original features: {df.shape[1]}")
print(f"Final features: {df_final.shape[1]}")
print(f"Features dropped: {df_eng.shape[1] - df_final.shape[1]}")
print(f"Final dataset shape: {df_final.shape}")

# Optional: See which features were dropped
dropped_features = [col for col in df_eng.columns if col not in final_feature_list]
print(f"\n=== DROPPED FEATURES ({len(dropped_features)}) ===")
for i, feature in enumerate(dropped_features[:10]):  # Show first 10
    print(f"{i+1}. {feature}")
if len(dropped_features) > 10:
    print(f"... and {len(dropped_features) - 10} more")


=== FEATURE ENGINEERING COMPLETE ===
Original features: 169
Final features: 91
Features dropped: 140
Final dataset shape: (69430, 91)

=== DROPPED FEATURES (140) ===
1. Data.ID
2. Latitude..N.S.
3. Longitude..E.W.
4. Location.source
5. Observation.period
6. Wheat.Type
7. Planting.date
8. Treatment
9. Treatment.type
10. Mean.annual.temperature..Â.C.
... and 130 more


In [57]:
df_final.to_csv('../../../data/finaldatasets/testdata/finalfr/wheat_final_features.csv', index=False)
print("Final dataset saved as 'wheat_final_features.csv'")

Final dataset saved as 'wheat_final_features.csv'


In [58]:
functions.summarize_dataframe(df_final)

| variable                                              | dtype   |   count |   pct_complete |   n_unique |
|-------------------------------------------------------|---------|---------|----------------|------------|
| Grain.yield..tons.ha.1.                               | float64 |   69430 |          100   |      14855 |
| evapS_total_season                                    | float64 |   69430 |          100   |       1155 |
| temp_grain_filling                                    | float64 |   69428 |          100   |        691 |
| water_use_efficiency_surface                          | float64 |   69430 |          100   |       1831 |
| precip_vegetative                                     | float64 |   69430 |          100   |        705 |
| LAI_duration_high                                     | int64   |   69430 |          100   |          6 |
| evap_total_season                                     | float64 |   69430 |          100   |        565 |
| water_stress_index        

In [59]:
new_order = [
	"Grain.yield..tons.ha.1.", "id", "Location", "Country", "Continent", "State.Region.County.Province",
	"Conversion.for.latitude", "Conversion.for.longitude", "Elevation", "AEZ", "end_date", "start_date", "year", "sowing_year", "Sand", "Silt", "Crop.variety",  # ← Added comma
	"Soil.pH", "Soil.organic.carbon..g.C.kg.1.", "Soil_N", "Tillage.type", "N.rate..kg.N.ha.1.", "P.rate..kg.P.ha.1.",  # ← Added comma
	"pr_irrigated", "Agricultural_Use_._Fungicides_and_Bactericides_.t.", "Agricultural_Use_._Herbicides_.t.",
	"Agricultural_Use_._Insecticides_.t.", "Use_per_area_of_cropland_._Pesticides_.total._.kg.ha.",
	"gdp_per_capita", "temp_above_optimal_grain_fill", "temp_critical_period", "temp_establishment",
	"temp_grain_filling", "temp_stem_elongation", "precip_establishment", "precip_grain_filling",
	"precip_reproductive", "precip_total_season", "precip_vegetative", "NDVI_area_under_curve",
	"NDVI_critical_avg", "NDVI_early_slope", "NDVI_grain_filling", "NDVI_late_slope", "NDVI_maintenance_ratio",
	"NDVI_max", "EVI_area_under_curve", "EVI_critical_avg", "EVI_early_slope", "EVI_grain_filling",
	"EVI_late_slope", "EVI_maintenance_ratio", "EVI_max", "LAI_decline_rate", "LAI_duration_high",
	"LAI_early_establishment", "LAI_early_growth_rate", "LAI_late_season", "LAI_max", "LAI_mid_season",
	"LAI_stay_green_index", "FPAR_area_under_curve", "FPAR_critical", "FPAR_early", "FPAR_maintenance_ratio",
	"FPAR_max", "soil_moisture_critical", "soil_moisture_establishment", "soil_moisture_grain_fill",
	"soil_moisture_seasonal_avg", "soil_moisture_variability", "soil_moisture_vegetative",
	"evapS_critical_period", "evapS_early_life", "evapS_total_season", "evap_critical_period",
	"evap_early_life", "evap_total_season", "transpiration_critical", "transpiration_early_stage",
	"transpiration_peak", "transpiration_total", "drought_stress_grain_fill", "num_heat_stress_months",
	"water_stress_index", "canopy_vigor_maintenance", "light_use_efficiency", "stay_green_index",
	"water_use_efficiency", "water_use_efficiency_surface", "biomass_accumulation_rate"
]

In [60]:
missing_cols = [col for col in new_order if col not in df_final.columns]
if missing_cols:
    print("Missing columns:", missing_cols)

In [61]:
df_final = df_final[new_order]

In [62]:
df_final.to_csv('../../../data/finaldatasets/testdata/finalfr/Betatree.csv', index=False)
print("Final dataset saved as 'Betatree.csv'")

Final dataset saved as 'Betatree.csv'
